# RTDL Quant Colab Setup

这个 Notebook 用于在 Google Colab 中准备并运行 RTDL Quant 项目。

默认假设你的 Google Drive 根目录下有：

- `alpha158_prices.parquet`
- `industry.csv`，可选

如果文件放在其他目录，修改下面的路径变量即可。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 克隆或更新项目

如果你已经 clone 过，下面的代码会自动 `git pull`。

In [ ]:
from pathlib import Path

repo_dir = Path('/content/rtdl-quant1')
if repo_dir.exists():
    %cd /content/rtdl-quant1
    !git pull
else:
    %cd /content
    !git clone https://github.com/xzteddy233/rtdl-quant1.git
    %cd /content/rtdl-quant1

## 2. 安装依赖

Colab 不需要创建 `.venv`，直接安装到当前环境。

In [ ]:
!pip install -r requirements.txt

## 3. 配置数据路径和训练参数

普通 Colab 建议先用 `LOAD_MAX_INSTRUMENTS = 500`。

如果遇到 `Killed`，改成 300 或 200。

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')

# 如果文件在 Drive 根目录，保持默认即可。
PARQUET_SOURCE = DRIVE_ROOT / 'alpha158_prices.parquet'
INDUSTRY_SOURCE = DRIVE_ROOT / 'industry.csv'

# 普通 Colab 推荐 500；如果内存不够，改成 300 或 200。
LOAD_MAX_INSTRUMENTS = 500

# auto: 有 industry + float_market_cap -> 完整中性化；只有 industry -> 行业中性化；否则关闭。
# 可选：'auto', 'full', 'industry', 'off'
NEUTRALIZATION = 'auto'

print('parquet:', PARQUET_SOURCE)
print('industry:', INDUSTRY_SOURCE)
print('load_max_instruments:', LOAD_MAX_INSTRUMENTS)
print('neutralization:', NEUTRALIZATION)

## 4. 复制数据、检查市值列、修改配置

这个 cell 会：

1. 复制 `alpha158_prices.parquet` 到 `data/`
2. 如果存在 `industry.csv`，复制到 `industry/`
3. 检查是否有 `float_market_cap`
4. 修改所有模型配置中的 `load_max_instruments`
5. 自动设置中性化

In [ ]:
import shutil
import pyarrow.parquet as pq
import yaml

LOCAL_PARQUET = Path('data/alpha158_prices.parquet')
LOCAL_INDUSTRY = Path('industry/industry.csv')

def copy_if_available(source: Path, target: Path, required: bool) -> bool:
    if not source.exists():
        print(f'missing: {source}')
        if required:
            raise FileNotFoundError(source)
        return False
    target.parent.mkdir(parents=True, exist_ok=True)
    print(f'copying {source} -> {target}')
    shutil.copy2(source, target)
    print(f'copied: {target} ({target.stat().st_size / 1024**3:.2f} GB)')
    return True

copy_if_available(PARQUET_SOURCE, LOCAL_PARQUET, required=True)
copy_if_available(INDUSTRY_SOURCE, LOCAL_INDUSTRY, required=False)

schema = pq.ParquetFile(LOCAL_PARQUET).schema_arrow
has_market_cap = 'float_market_cap' in schema.names
has_industry = LOCAL_INDUSTRY.exists()

print('has float_market_cap:', has_market_cap)
print('has industry.csv:', has_industry)
print('last columns:', schema.names[-10:])

if NEUTRALIZATION == 'auto':
    if has_industry and has_market_cap:
        neutralization_mode = 'full'
    elif has_industry:
        neutralization_mode = 'industry'
    else:
        neutralization_mode = 'off'
else:
    neutralization_mode = NEUTRALIZATION

if neutralization_mode == 'full' and not has_market_cap:
    raise ValueError('full neutralization requires float_market_cap')
if neutralization_mode in {'full', 'industry'} and not has_industry:
    raise ValueError('industry neutralization requires industry.csv')

for path in sorted(Path('rtdl_quant/configs').glob('*.yaml')):
    config = yaml.safe_load(path.read_text()) or {}
    config.setdefault('data', {})['load_max_instruments'] = LOAD_MAX_INSTRUMENTS
    n = config.setdefault('evaluation', {}).setdefault('neutralization', {})
    if neutralization_mode == 'off':
        n['enabled'] = False
    elif neutralization_mode == 'industry':
        n['enabled'] = True
        n['industry'] = True
        n['market_cap'] = False
        n['standardize'] = True
    elif neutralization_mode == 'full':
        n['enabled'] = True
        n['industry'] = True
        n['market_cap'] = True
        n['standardize'] = True
    else:
        raise ValueError(neutralization_mode)
    path.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))
    print(f'updated {path}: load_max_instruments={LOAD_MAX_INSTRUMENTS}, neutralization={neutralization_mode}')

## 5. 检查 GPU

MLP/ResNet/FTT 建议开 GPU；CatBoost/XGBoost 可以先用 CPU。

In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    !nvidia-smi

## 6. 训练模型

建议先跑 MLP，确认流程正常后再跑其他模型。

In [ ]:
!python -m rtdl_quant.main --config rtdl_quant/configs/mlp.yaml

In [ ]:
# 可选：继续训练其他模型
# !python -m rtdl_quant.main --config rtdl_quant/configs/resnet.yaml
# !python -m rtdl_quant.main --config rtdl_quant/configs/ft_transformer.yaml
# !python -m rtdl_quant.main --config rtdl_quant/configs/catboost.yaml
# !python -m rtdl_quant.main --config rtdl_quant/configs/xgboost.yaml

## 7. 查看结果

In [ ]:
import pandas as pd

pd.read_csv('rtdl_quant/outputs/alpha158_mlp/metrics.csv')

In [ ]:
# 如果已经训练多个模型，可以生成总对比表
!python -m rtdl_quant.scripts.compare_models
pd.read_csv('rtdl_quant/outputs/model_comparison.csv')

## 8. 保存结果到 Google Drive

Colab 重启后本地文件会丢，训练后建议保存。

In [ ]:
!rm -rf '/content/drive/MyDrive/rtdl_outputs'
!cp -r rtdl_quant/outputs '/content/drive/MyDrive/rtdl_outputs'
print('saved to /content/drive/MyDrive/rtdl_outputs')